# 05 — Advanced data-science laboratories
Each section answers a different research question. These demonstrations do not claim that every modality is already part of the production forecast.

In [ ]:
%pip install -q -e .
import numpy as np
import pandas as pd
from agridecision.advanced.clustering import build_market_profiles, segment_markets
from agridecision.advanced.graph import correlation_edges, market_centrality
from agridecision.advanced.nlp import make_bulletin_risk_model, top_explanatory_terms
from agridecision.advanced.satellite import ndvi, summarise_field_index
from agridecision.advanced.simulation import simulate_market_policy
from agridecision.advanced.survival import extract_shock_episodes, kaplan_meier
from agridecision.monitoring.drift import numeric_drift_report

In [ ]:
prices = pd.read_csv('data/processed/mandi_prices.csv', parse_dates=['arrival_date'])
profiles = build_market_profiles(prices)
segments, cluster_model = segment_markets(profiles, clusters=min(3, len(profiles)))
segments.sort_values(['cluster', 'mean_price'])

In [ ]:
edges = correlation_edges(prices, minimum_absolute_correlation=0.40)
print(edges.head())
market_centrality(edges).head(10)

## NLP demonstration
Replace these illustrative labelled sentences with dated, licensed official bulletins before evaluation.

In [ ]:
bulletins = [
    'Heavy rainfall damaged onion storage and disrupted market arrivals',
    'Normal arrivals and stable weather were reported across markets',
    'Road flooding delayed crop transport and reduced supply',
    'Harvest progress is normal with adequate market supply',
]
labels = [1, 0, 1, 0]
nlp_model = make_bulletin_risk_model().fit(bulletins, labels)
print(nlp_model.predict_proba(['Flooding may disrupt onion arrivals']))
top_explanatory_terms(nlp_model, count=5)

## Satellite index demonstration
The arrays below are spectral examples, not downloaded field imagery.

In [ ]:
nir = np.array([[0.8, 0.7], [0.3, 0.5]])
red = np.array([[0.2, 0.3], [0.3, 0.2]])
vegetation = ndvi(nir, red)
print(vegetation)
summarise_field_index(vegetation)

In [ ]:
working = prices.sort_values(['market', 'arrival_date']).copy()
working['return_7d'] = working.groupby('market')['modal_price'].pct_change(7)
working['is_shock'] = working['return_7d'] >= 0.15
episodes = extract_shock_episodes(working)
if not episodes.empty:
    display(kaplan_meier(episodes['duration_days'], episodes['event_observed']))

In [ ]:
history, policy = simulate_market_policy({'Market A': 1500, 'Market B': 1580, 'Market C': 1530}, steps=365)
print(dict(zip(policy.market_names, policy.counts)))
history.tail()

In [ ]:
split_date = prices['arrival_date'].quantile(0.8)
reference = prices[prices['arrival_date'] < split_date]
current = prices[prices['arrival_date'] >= split_date]
numeric_drift_report(reference, current, ['modal_price', 'min_price', 'max_price'])